# 09 - Model Governance and Monitoring

This notebook converts the modelling, threshold-selection, and explainability outputs into governance-ready documentation.

The goal is to show how a Canadian retail credit-risk analytics project would be controlled after model development:

- model inventory and intended use
- validation/test performance summary
- leakage and fairness/proxy controls
- explainability governance
- operating threshold controls
- monitoring KPIs and escalation limits
- model card, validation summary, stakeholder brief, and monitoring plan

## Professional framing

A credit-risk model is not complete when the ROC-AUC or PR-AUC is calculated. For a banking-style portfolio project, the model must also be explainable, monitored, and governed.

This notebook positions the model as a **decision-support and manual-review prioritization tool**, not an automated credit-decline engine. That distinction is important because the dataset and project do not include all production controls required for automated lending decisions.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from credit_risk.config import GOVERNANCE_DIR, PROCESSED_DIR, TABLE_DIR, ensure_project_directories
from credit_risk.governance.model_governance import (
    build_control_register,
    build_feature_governance_summary,
    build_governance_summary,
    build_model_inventory,
    build_model_risk_limit_register,
    build_monitoring_kpi_snapshot,
    build_validation_test_summary,
    build_xai_governance_summary,
    load_governance_inputs,
    save_governance_outputs,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
ensure_project_directories()

## Load governance inputs

This step reads outputs produced by earlier notebooks:

- modelling dataset and feature catalogue from Notebook 05
- model results from Notebook 06
- threshold outputs from Notebook 07
- XAI outputs from Notebook 08

In [2]:
inputs = load_governance_inputs(processed_dir=PROCESSED_DIR, table_dir=TABLE_DIR)

{
    "modeling_rows": len(inputs.modeling_df),
    "modeling_columns": inputs.modeling_df.shape[1],
    "feature_catalog_rows": len(inputs.feature_catalog),
    "model_selection_rows": len(inputs.model_selection),
    "threshold_summary_rows": len(inputs.recommended_threshold),
    "xai_grouped_importance_rows": len(inputs.xai_grouped_importance),
}

{'modeling_rows': 134417,
 'modeling_columns': 51,
 'feature_catalog_rows': 47,
 'model_selection_rows': 3,
 'threshold_summary_rows': 1,
 'xai_grouped_importance_rows': 47}

## Model inventory

The inventory table defines what the model is, what it is used for, and the key governance boundaries.

In [3]:
model_inventory = build_model_inventory(inputs)
model_inventory

,item,value
0,business_use,Early-warning retail credit default-risk ranking and manual-review prioritization
1,target,Defaulter indicator
2,champion_model,xgboost_weighted
3,operating_threshold,0.565
4,threshold_objective,minimum_cost_review_rate_le_30pct
5,modeling_rows,134417
6,portfolio_default_rate,0.090413
7,model_feature_count,47
8,sensitive_proxy_use,Excluded from baseline model; retained only for audit/governance review where permitted
9,leakage_control,Repayment-derived variables excluded from model features


## Validation, test, and operating-threshold summary

The table compares the original default 0.50 model output with the selected business operating threshold. The selected threshold is the relevant view for operational review planning.

In [4]:
validation_test_summary = build_validation_test_summary(inputs)
validation_test_summary

,evaluation_view,model_name,dataset,threshold,roc_auc,pr_auc,brier_score,recall,precision,f1,balanced_accuracy,mcc,review_rate,business_cost,false_negative,false_positive,true_positive,true_negative,selected_operating_threshold
0,validation_default_0_50,xgboost_weighted,validation,0.500,0.751101,0.229393,0.204092,0.720790,0.171429,0.276981,0.687249,0.221241,0.380152,5720500.0,509,6351,1314,11989,False
1,test_default_0_50,xgboost_weighted,test,0.500,0.746768,0.216785,0.203494,0.727372,0.172904,0.279393,0.690758,0.225365,0.380350,5656500.0,497,6343,1326,11997,False
2,validation_selected_operating_threshold,xgboost_weighted,validation,0.565,0.751101,0.229393,0.204092,0.619309,0.189620,0.290343,0.678111,0.223938,0.295293,5882500.0,694,4825,1129,13515,True
3,test_selected_operating_threshold,xgboost_weighted,test,0.565,0.746768,0.216785,0.203494,0.614920,0.190971,0.291434,0.677989,0.224717,0.291127,5884500.0,702,4749,1121,13591,True


## Feature governance summary

This summarizes which variables are used, excluded, or retained only for monitoring/governance review. The most important control is that repayment-derived variables are not used as model predictors.

In [5]:
feature_governance_summary = build_feature_governance_summary(inputs)
feature_governance_summary

,feature_count
0,64


## Explainability governance

The XAI summary converts SHAP drivers into governance notes. Data-quality and pricing-related drivers are useful but require careful interpretation.

In [6]:
xai_governance_summary = build_xai_governance_summary(inputs, top_n=12)
xai_governance_summary

,raw_feature,feature_label,mean_abs_shap,mean_shap,transformed_feature_count,governance_note
0,interest_rate,Interest Rate,0.618882,-0.054802,1,Pricing/risk signal: explain carefully because pricing can reflect prior underwriting risk.
1,amount_missing_flag,Amount Missing Flag,0.358383,0.080368,1,Data-quality signal: monitor for drift and do not interpret as borrower behaviour alone.
2,broad_data_quality_issue_count,Broad Data Quality Issue Count,0.170565,-0.050188,1,Data-quality signal: monitor for drift and do not interpret as borrower behaviour alone.
3,total_income_pa,Total Income Pa,0.138081,-0.019256,1,Affordability/exposure signal: suitable for portfolio-risk interpretation.
4,core_data_quality_issue_count,Core Data Quality Issue Count,0.085640,-0.065911,1,Data-quality signal: monitor for drift and do not interpret as borrower behaviour alone.
5,loan_to_income_band,Loan To Income Band,0.073130,0.023619,1,Affordability/exposure signal: suitable for portfolio-risk interpretation.
6,high_interest_flag,High Interest Flag,0.072073,0.014624,1,Pricing/risk signal: explain carefully because pricing can reflect prior underwriting risk.
7,tenure_years,Tenure Years,0.059983,-0.011390,1,Segment/product signal: monitor segment stability and business reasonableness.
8,loan_category,Loan Category,0.053363,0.000129,1,Segment/product signal: monitor segment stability and business reasonableness.
9,amount,Amount,0.043531,-0.016452,1,Affordability/exposure signal: suitable for portfolio-risk interpretation.


## Control register

This table is the project's model-risk control register. It links each risk to a control, evidence source, owner, and review cadence.

In [7]:
control_register = build_control_register()
control_register

,control_area,risk,control,evidence,owner,frequency
0,Data ingestion,Many-to-many sheet merge inflates borrower records and changes target rate.,Merge sheets using user_id plus record_sequence and test duplicate record keys.,Notebook 01 ingestion summary and record-key duplicate check.,Risk analytics,Each data refresh
1,Data quality,Missing amount/employment fields may distort score interpretation.,"Create missingness flags, monitor missingness rates, and retain data-quality features only where justified.",Notebook 02/03 missingness and data-quality flag summaries.,Data analyst / data steward,Monthly
2,Leakage prevention,Repayment-derived variables can leak target information into model training.,"Exclude total_payment, received_principal, interest_received, and derived repayment ratios from model features.",Feature leakage and usage policy table.,Model developer,Before each model release
3,Fairness and proxy risk,Sensitive/proxy variables could create unfair or hard-to-defend decisions.,"Exclude gender, marital status, pincode, and social profile from baseline model; retain only for approved audit review.",Feature policy and model feature catalogue.,Model governance / compliance partner,Before release and annually
4,Model performance,Model ranking weakens or becomes unstable after deployment.,"Monitor PR-AUC, ROC-AUC, recall, precision, Brier score, and confusion-matrix outcomes when labels mature.",Notebook 06 validation/test results.,Model monitoring team,Monthly/quarterly
5,Threshold governance,Operating threshold overloads manual review teams or misses too many defaults.,Select threshold on validation data using business-cost and review-cap constraints; validate once on test data.,Notebook 07 threshold shortlist and recommended threshold summary.,Credit-risk strategy,Quarterly or after material portfolio change
6,Explainability,Stakeholders cannot understand why accounts are sent to review.,"Provide global SHAP drivers, local reason codes, anchor-like rules, and counterfactual scenario diagnostics.",Notebook 08 XAI outputs.,Risk analytics / model validation,Each model release
7,Ongoing monitoring,"Population, score, or data-quality drift changes model behaviour.","Track score distribution, review rate, feature missingness, top-driver PSI, and realized default outcomes.",Notebook 09 monitoring plan and risk-limit register.,Model owner,Monthly


## Monitoring KPI snapshot

This is the first monitoring baseline for the model. In a real deployment, the same KPIs would be refreshed every month or after each material data refresh.

In [8]:
monitoring_kpi_snapshot = build_monitoring_kpi_snapshot(inputs)
monitoring_kpi_snapshot

,kpi,value,interpretation
0,Champion model,xgboost_weighted,Selected using validation ranking metrics.
1,Portfolio default rate,9.04%,Base rate for interpreting precision and review workload.
2,Recommended objective,minimum_cost_review_rate_le_30pct,Threshold-selection business rule.
3,Operating threshold,0.565,Probability cutoff for manual-review flag.
4,Test ROC-AUC,0.7468,Out-of-sample ranking performance.
5,Test PR-AUC,0.2168,Ranking performance under class imbalance.
6,Validation recall at operating threshold,61.93%,Defaults captured during threshold selection.
7,Test recall at operating threshold,61.49%,Out-of-sample defaults captured at selected threshold.
8,Test precision at operating threshold,19.10%,Share of reviewed accounts that defaulted.
9,Test review rate at operating threshold,29.11%,Operational workload from the score cutoff.


## Model risk limits and escalation triggers

The limits below are practical monitoring thresholds for a portfolio project. They are intentionally labelled as monitoring triggers, not regulatory limits.

In [9]:
risk_limit_register = build_model_risk_limit_register(inputs)
risk_limit_register

,metric,baseline,warning_limit,breach_limit,monitoring_frequency,action
0,Population Stability Index - overall score,Training/test score distribution,> 0.10,> 0.25,Monthly,"Investigate portfolio shift, data pipeline changes, and score calibration."
1,Key feature PSI - top SHAP drivers,Training/test feature distributions,> 0.10,> 0.25,Monthly,Review feature distribution shift and retraining need.
2,Operating review rate,29.11%,> 34.11%,> 39.11%,Weekly/monthly,Review threshold capacity and manual-review staffing impact.
3,Recall on matured labels,61.49%,< 56.49%,< 51.49%,Monthly after labels mature,Assess missed-default concentration and model refresh need.
4,Precision on reviewed accounts,19.10%,< 16.10%,< 14.10%,Monthly after labels mature,Review false-positive burden and customer-impact risk.
5,Critical data-quality missingness,Notebook 02/03 data-quality profile,+25% relative increase,+50% relative increase,Each data load,Open data-quality incident and pause model refresh if severe.


## One-row governance summary

This table is suitable for the README, stakeholder deck, or interview explanation.

In [10]:
governance_summary = build_governance_summary(inputs)
governance_summary

,champion_model,modeling_rows,portfolio_default_rate,operating_threshold,threshold_objective,test_recall,test_precision,test_review_rate,test_business_cost,primary_governance_decision
0,xgboost_weighted,134417,0.090413,0.565,minimum_cost_review_rate_le_30pct,0.61492,0.190971,0.291127,5884500.0,"Use as decision-support/manual-review prioritization model, not as an automated credit-decline engine."


## Save governance tables and documentation

This step creates CSV tables plus markdown documents under `reports/governance/`.

In [13]:
saved_outputs = save_governance_outputs(
    inputs=inputs,
    table_dir=TABLE_DIR,
    governance_dir=GOVERNANCE_DIR,
)

saved_outputs

{'model_inventory': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\model_inventory.csv',
 'model_validation_test_summary': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\model_validation_test_summary.csv',
 'feature_governance_summary': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\feature_governance_summary.csv',
 'xai_governance_summary': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\xai_governance_summary.csv',
 'model_control_register': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\model_control_register.csv',
 'model_risk_limit_register': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\model_risk_limit_register.csv',
 'model_monitoring_kpi_snapshot': 'D:\\Banking and Finance\\Projects\\canadian-retail-credit-risk-xai\\reports\\tables\\model_monitori

## Preview stakeholder brief

The stakeholder brief is the non-technical explanation you can show in GitHub or discuss during interviews.

In [14]:
stakeholder_brief_path = GOVERNANCE_DIR / "stakeholder_brief.md"
print(stakeholder_brief_path.read_text(encoding="utf-8")[:2500])

# Stakeholder Brief - Retail Credit Default-Risk Model

## What the model does

The model ranks borrowers by estimated default risk so that a credit-risk team can prioritize manual review and portfolio monitoring.

## Recommended operating point

- **Champion model:** xgboost_weighted
- **Operating threshold:** 0.565
- **Test recall:** 61.49%
- **Test precision:** 19.10%
- **Test review rate:** 29.11%

## Business interpretation

At the selected threshold, the model captures a meaningful share of future defaults while keeping the review population below the operational cap used in this project. This makes it more practical than using the default 0.50 cutoff.

## Main risk drivers identified by explainability

| raw_feature                    | feature_label                  |   mean_abs_shap |   mean_shap |   transformed_feature_count | governance_note                                                                             |
|:-------------------------------|:----------------------

## Notebook 09 conclusions

Carry these decisions into the final README and portfolio summary:

1. The model is positioned as a **manual-review prioritization tool**, not an automated decline engine.
2. The selected threshold balances default capture and review-team workload.
3. Leakage controls exclude repayment-derived predictors from model training.
4. Sensitive/proxy-sensitive variables are excluded from the baseline model.
5. SHAP, anchor-like rules, and counterfactual diagnostics provide explainability evidence.
6. The project now includes a model card, validation summary, stakeholder brief, monitoring plan, control register, and risk-limit register.